# Ejecución de inferencia y harness

## 1. Clonar repositorio

In [1]:
# Clonar el repo o actualizar si ya existe (rama Luis)
import os
if not os.path.exists("/content/corpus"):
    !git clone --branch Luis --single-branch https://github.com/luisNP21/Topicos-IA.git /content/corpus
else:
    %cd /content/corpus
    !git fetch origin Luis
    !git checkout Luis
    !git pull origin Luis

Cloning into '/content/corpus'...
remote: Enumerating objects: 1596, done.
remote: Counting objects: 100% (322/322), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 1596 (delta 187), reused 207 (delta 156), pack-reused 1274 (from 1)
Receiving objects: 100% (1596/1596), 33.40 MiB | 36.94 MiB/s, done.
Resolving deltas: 100% (717/717), done.


In [2]:
%cd /content/corpus
%ls

/content/corpus
M1/  M2/  M3/  M4/  README.md  review/


## 2. Instalar dependencias

In [3]:
!pip install -r M2/run_inference/requirements.txt
!pip install -r M2/harness/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 23.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 95.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 96.6 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.21.0
    Uninstalling transformers-5.21.0:
      Successfully uninstalled transformers-5.21.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.15.0
    Uninstalling accelerate-1.15.0:
      Successfully uninstalled accelerate-1.15.0
  Attempting uninstall: peft
    Found existing installation: peft 0.21.2
    Uninstalling peft-0.21.2:
      Successfully uninstalled peft-0.21.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [peft]
  Attempting uninstall: python-dotenv
    Found existing inst

## 3. Configuración de paths y checks de recursos necesitados

In [4]:
from google.colab import drive
drive.mount("/content/drive")

import json
from pathlib import Path
import shutil
import yaml

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")

# 1. Detectar o asegurar la clonación del repositorio (rama Luis)
candidatos_repo = [
    Path("/content/corpus"),
    Path("/content/Topicos-IA"),
    Path.cwd(),
]
REPO = next((c for c in candidatos_repo if (c / "M2/run_inference/config.yaml").exists()), None)
if REPO is None:
    !git clone --branch Luis --single-branch https://github.com/luisNP21/Topicos-IA.git /content/corpus
    REPO = Path("/content/corpus")

# 2. Gold Set de los 15 casos (texto clínico original): se copia SIEMPRE desde el repo,
#    para que el baseline de texto use el mismo gold que las corridas de visión. Si ya
#    había un gold_casos.jsonl en Drive (de una corrida anterior), se reemplaza.
gold_src = REPO / "M4/eval_set/eval_set_casos.jsonl"
assert gold_src.exists(), f"No existe {gold_src}. Revisar que la rama clonada lo tenga."
gold_dest = PROJECT_ROOT / "M2/eval_harness/gold_casos.jsonl"
gold_dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(gold_src, gold_dest)
_gold = [json.loads(l) for l in open(gold_dest, encoding="utf-8") if l.strip()]
print(f"Gold copiado desde el repo: {len(_gold)} casos, "
      f"{sum(len(g.get('entities_gold') or []) for g in _gold)} entidades -> {gold_dest}")

checks = {
    "Adaptador LoRA": PROJECT_ROOT / "M1/saved_models/clinical_bert-distemist-lora/adapter_model.safetensors",
    "Gold set": gold_dest,
}
for nombre, path in checks.items():
    print(f"{nombre}: {'OK' if path.exists() else 'FALTA'} -- {path}")

# 3. Configurar el harness normal (modo texto, sin visión) sobre los 15 casos
gold_rel = str(gold_dest.relative_to(PROJECT_ROOT)).replace("\\", "/")

# Config de inferencia
ri_cfg_path = REPO / "M2/run_inference/config.yaml"
with open(ri_cfg_path, "r", encoding="utf-8") as f:
    ri_cfg = yaml.safe_load(f)
ri_cfg["eval_set_path"] = gold_rel
ri_cfg["fuente"] = {"tipo": "texto"}  # Texto directo original, sin visión
ri_cfg["output_path"] = "M2/predictions/encoder_solo.json"
with open(ri_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(ri_cfg, f, sort_keys=False)

# Config del harness
h_cfg_path = REPO / "M2/harness/config.yaml"
with open(h_cfg_path, "r", encoding="utf-8") as f:
    h_cfg = yaml.safe_load(f)
h_cfg["rutas"]["gold_set"] = gold_rel
h_cfg["rutas"]["outputs_dir"] = "M2/outputs"
with open(h_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(h_cfg, f, sort_keys=False)

print(f"\nRepo ubicado en: {REPO}")
print(f"Configs actualizados: eval_set_path = {gold_rel}")

Mounted at /content/drive
Gold copiado desde el repo: 15 casos, 57 entidades -> /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_casos.jsonl
Adaptador LoRA: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M1/saved_models/clinical_bert-distemist-lora/adapter_model.safetensors
Gold set: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_casos.jsonl

Repo ubicado en: /content/corpus
Configs actualizados: eval_set_path = M2/eval_harness/gold_casos.jsonl


In [5]:
from pathlib import Path
from google.colab import userdata

with open(REPO / ".env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")
    f.write(f"GROQ_API_KEY={userdata.get('GROQ_API_KEY')}\n")
print(f".env generado en {REPO / '.env'}")

.env generado en /content/corpus/.env


## 4. Correr inferencia

In [6]:
%cd M2/run_inference/

/content/corpus/M2/run_inference


In [7]:
!python run_inference.py

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu130 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
config.json: 100% 613/613 [00:00<00:00, 2.91MB/s]

pytorch_model.bin: downloading bytes:  61% 305M/504M [00:01<00:00, 220MB/s, 26.0MB/s  ]
pytorch_model.bin: downloading bytes:  66% 334M/504M [00:02<00:01, 109MB/s, 28.8MB/s  ]
pytorch_model.bin: downloading bytes: 100% 335M/335M [00:03<00:00, 107MB/s, 28.6MB/s  ]
pytorch_model.bin: reconstructing file: 100% 504M/504M [00:03<00:00, 160MB/s, 46.1MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 16347.37it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        |

## 5. Correr harness

In [8]:
%cd ..

/content/corpus/M2


In [9]:
%cd harness
%ls

/content/corpus/M2/harness
cached_system.py    harness.py              metrics_semantic.py  scorecard.py
config_vision.yaml  length_bias_pairs.json  __pycache__/
config.yaml         metrics_exact.py        requirements.txt
gold_loader.py      metrics_judge.py        run_harness.py


In [10]:
!python run_harness.py

Proyecto en: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
Gold set: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_casos.jsonl | existe: True
Predicciones: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/predictions/encoder_solo.json | existe: True
Output: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/outputs
Gold set cargado: 15 ejemplos
Formato valido
Primer ejemplo:
  text[:120]: Varón de 52 años, con sobrepeso (IMC 29.3 kg/m2) y vida sedentaria, que acude a consulta de medicina preventiva. En anal ...
  entities_gold: ['sobrepeso', 'enfermedad cardiovascular', 'diabetes mellitus tipo 2']
  n_entidades: 3
[21:04:33] Seed global fijada: 42
[21:04:33] PROJECT_ROOT: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
[21:04:33] DIMENSION 1 -- Exact-match
Proyecto en: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
Gold set: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_casos.jsonl | existe: True
Predicciones: /content/drive/MyDrive/To